# 00 — Setup check

Confirms the demo stack is healthy: SparkSession connects to the standalone master, S3A talks to MinIO, and the Delta + Iceberg libraries load.

Run **Cell → Run All**. Every cell should finish without errors.

In [ ]:
from spark_session import get_spark

spark = get_spark("00-setup-check")
spark

In [ ]:
import importlib.metadata as md

print("Spark      :", spark.version)
print("Hadoop     :", spark._jvm.org.apache.hadoop.util.VersionInfo.getVersion())
print("Delta      :", md.version("delta-spark"))
print("PyIceberg  :", md.version("pyiceberg"))
print("Master URL :", spark.sparkContext.master)

## S3A round-trip

Writes one row through S3A → MinIO and reads it back. If this works, every later notebook will work too.

In [ ]:
path = "s3a://raw-data/setup-check/"
df = spark.createDataFrame([(1, "hello"), (2, "world")], ["id", "msg"])
df.write.mode("overwrite").parquet(path)
spark.read.parquet(path).show()

In [ ]:
# List MinIO buckets via boto3 — proves credentials end-to-end.
import os, boto3
s3 = boto3.client(
    "s3",
    endpoint_url=os.environ.get("S3_ENDPOINT", "http://minio:9000"),
    aws_access_key_id=os.environ["MINIO_ACCESS_KEY"],
    aws_secret_access_key=os.environ["MINIO_SECRET_KEY"],
)
[b["Name"] for b in s3.list_buckets()["Buckets"]]

## Optional — submitting to a remote Kubernetes cluster

The block below is **commented out** because it needs a real K8s cluster + an image you've built and pushed. Use it as a template; for the local demo you can keep using `spark://spark-master:7077`.

```python
# from pyspark.sql import SparkSession
# spark_k8s = (
#     SparkSession.builder
#         .master("k8s://https://<K8S_API>:6443")
#         .appName("k8s-demo")
#         .config("spark.submit.deployMode", "cluster")
#         .config("spark.kubernetes.container.image", "ghcr.io/my-org/spark:3.5.3")
#         .config("spark.kubernetes.namespace", "default")
#         .config("spark.kubernetes.authenticate.driver.serviceAccountName", "spark")
#         .config("spark.hadoop.fs.s3a.endpoint", "http://minio.minio.svc:9000")
#         .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
#         .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
#         .config("spark.hadoop.fs.s3a.path.style.access", "true")
#         .getOrCreate()
# )
```

See `scripts/k8s-spark-submit.sh` for the command-line equivalent.